## Import libraries

In [32]:
import pandas as pd
import re
import gget
from tqdm import tqdm

## Load datset

extract **stage, history diagnosis** from the matrix file

In [66]:
# Path to the uploaded file
file_path = "GSE81089_series_matrix.txt"

# Read the file content
with open(file_path, "r") as file:
    lines = file.readlines()

# Extract stage tnm and histology data
stage_data = []
histology_data = []
samples = []

for line in lines:
    if line.startswith("!Sample_title"):
        samples = re.findall(r'"(.*?)"', line)
    if "stage tnm" in line:
        stage_data = re.findall(r'"stage tnm: (\d+)"', line)
    elif "histology" in line:
        histology_data = re.findall(r'"histology: (\d+)"', line)


# Ensure both lists have the same length
min_length = min(len(stage_data), len(histology_data))
stage_data = stage_data[:min_length]
histology_data = histology_data[:min_length]
samples = samples[:min_length]

# Create a DataFrame
df_matrix = pd.DataFrame({"sample": samples, "stage": stage_data, "diagnosis": histology_data})
print(f"# of rows: {min_length}")
print(df_matrix.head())

# of rows: 199
  sample stage diagnosis
0  L400T     3         2
1  L401T     5         2
2  L404T     3         2
3  L406T     1         1
4  L413T     5         2


extract gene expression 

ref: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE81089

In [ ]:
# Specify the CSV file path
csv_file = "GSE81089_FPKM_cufflinks.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Select only the columns that match the list
samples.append("Ensembl_gene_id") # Add the gene id
cols_drop = [col for col in df.columns if col not in samples]
filtered_df = df.drop(cols_drop, axis=1)
filtered_df = filtered_df.T
print(filtered_df.head())

   Ensembl_gene_id      L400T      L401T      L404T     L406T     L413T  \
0  ENSG00000000003  52.195000  37.889100  23.191000  25.03240  41.96860   
1  ENSG00000000005   0.230061   0.086034   0.048022   0.00000   2.57090   
2  ENSG00000000419  43.861600  47.045700  38.129200  54.30300  51.29690   
3  ENSG00000000457  14.710100   7.812330  12.311700   8.41631   8.84999   
4  ENSG00000000460   4.813350   5.920730   8.213850   6.71221   4.79088   

       L414T      L417T     L420T     L439T  ...      L876T      L877T  \
0  28.579400  19.900800  23.26320  37.02400  ...  15.192200  29.719400   
1   0.087192   0.234047   0.00000   0.00000  ...   0.187195   0.404797   
2  42.160400  78.196100  60.72830  23.52960  ...  27.456400  35.623900   
3   6.280520   4.662350   7.39264  14.55440  ...   7.622990  10.079700   
4   9.647640   6.605090   8.49746   5.45051  ...   3.631520   6.944480   

      L879T     L880T     L881T      L884T     L885T     L886T     L887T  \
0  28.03980  21.30900  22.72

merge matrix data and gene expression by samples' id

In [91]:
df1 = df_matrix
df2 = filtered_df.set_index("Ensembl_gene_id").T 

# Merge the DataFrames on the sample column (index of df2)
merged_df = df1.merge(df2, left_on="sample", right_index=True)
print(f"total rows: {len(merged_df)}")
print(merged_df.head())

total rows: 199
  sample stage diagnosis  ENSG00000000003  ENSG00000000005  ENSG00000000419  \
0  L400T     3         2          52.1950         0.230061          43.8616   
1  L401T     5         2          37.8891         0.086034          47.0457   
2  L404T     3         2          23.1910         0.048022          38.1292   
3  L406T     1         1          25.0324         0.000000          54.3030   
4  L413T     5         2          41.9686         2.570900          51.2969   

   ENSG00000000457  ENSG00000000460  ENSG00000000938  ENSG00000000971  ...  \
0         14.71010          4.81335          7.40831         112.4260  ...   
1          7.81233          5.92073          9.83188          39.7146  ...   
2         12.31170          8.21385          9.68575          25.9596  ...   
3          8.41631          6.71221         10.92630          80.2073  ...   
4          8.84999          4.79088          8.36149          38.4429  ...   

   ENSG00000272537  ENSG00000272538  ENS

In [ ]:
# Load the GTF file (change path to your actual file)
gtf_file = "Homo_sapiens.GRCh38.77.gtf"

# Read GTF and extract gene_id and gene_name
gtf_data = pd.read_csv(gtf_file, comment='#', sep='\t', header=None)

# Extract relevant columns
attributes = gtf_data.iloc[:, 8]

# Parse gene_id and gene_name from the attributes column
gene_map = {}
for attr in attributes:
    gene_id_match = re.search(r'gene_id "([^"]+)"', attr)
    gene_name_match = re.search(r'gene_name "([^"]+)"', attr)
    
    if gene_id_match and gene_name_match:
        gene_id = gene_id_match.group(1)
        gene_name = gene_name_match.group(1)
        gene_map[gene_id] = gene_name

# Convert to DataFrame
gene_df = pd.DataFrame(list(gene_map.items()), columns=["gene_id", "gene_name"])
print(gene_df.head())

           gene_id   gene_name
0  ENSG00000223972     DDX11L1
1  ENSG00000227232      WASH7P
2  ENSG00000278267   MIR6859-2
3  ENSG00000243485  MIR1302-10
4  ENSG00000274890  MIR1302-11


In [ ]:
# List of gene IDs to convert (replace with your actual IDs)
gene_ids = df["Ensembl_gene_id"].to_list() 
print(f"# of gene IDs: {len(gene_ids)}")

# Query Ensembl for gene names
miss_gene_names = []
gene_names = []
for id in gene_ids:
    try:
        name = gene_df[id]
        gene_names.append(name)
    except:
        miss_gene_names.append(id)

# of gene IDs: 63130
